In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
try:
    df = pd.read_csv('ALL_UQ_PREDICTED.csv')
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
except FileNotFoundError:
    print("Error: 'ALL_UQ_PREDICTED.csv' not found. Please upload the file.")
    # Create dummy structure to prevent crash if file missing
    df = pd.DataFrame(columns=['date', 'actual']) 
    if not df.empty: df.set_index('date', inplace=True)

# ==========================================
# 2. COLUMN PARSING
# ==========================================
# We need to identify the "base" prediction columns vs the bounds (_L, _U)
# Base columns are those that do NOT end in _L or _U
all_cols = [c for c in df.columns if c != 'actual']
base_cols = [c for c in all_cols if not c.endswith('_L') and not c.endswith('_U')]

# Extract Metadata
models = set()
schemes = set()
seeds = set()

col_meta = {} # Map base_col -> (model, scheme, seed)

for col in base_cols:
    parts = col.split('_')
    # Assumes format: model_scheme_seed (e.g., gru_mcd_42)
    # This handles names like gru_mcd_1234 correctly
    if len(parts) >= 3:
        m, s, sd = parts[0], parts[1], parts[2]
        models.add(m)
        schemes.add(s)
        seeds.add(sd)
        col_meta[col] = (m, s, sd)

sorted_models = sorted(list(models))
sorted_schemes = sorted(list(schemes))
sorted_seeds = sorted(list(seeds), key=lambda x: int(x) if x.isdigit() else x)

# ==========================================
# 3. WIDGETS
# ==========================================
style = {'description_width': 'initial'}

w_model = widgets.Dropdown(options=['All'] + sorted_models, value='All', description='Model:', style=style)
w_scheme = widgets.Dropdown(options=['All'] + sorted_schemes, value='All', description='UQ Scheme:', style=style)
w_seed = widgets.Dropdown(options=['All'] + sorted_seeds, value='All', description='Seed:', style=style)
w_show_band = widgets.Checkbox(value=True, description='Show Uncertainty Bands')

# Date Slider
dates = df.index
if len(dates) > 0:
    w_range = widgets.IntRangeSlider(
        value=[0, len(dates)-1],
        min=0, max=len(dates)-1, step=1,
        description='Date Zoom:',
        continuous_update=False,
        layout=widgets.Layout(width='95%')
    )
else:
    w_range = widgets.IntRangeSlider(min=0, max=1)

# ==========================================
# 4. PLOTTING LOGIC
# ==========================================
def plot_uq_series(model, scheme, seed, idx_range, show_band):
    if df.empty:
        print("No data available.")
        return

    start_idx, end_idx = idx_range
    sub_df = df.iloc[start_idx : end_idx+1]
    
    # Filter columns to plot based on dropdown selection
    cols_to_plot = []
    for col in base_cols:
        m, s, sd = col_meta.get(col, (None, None, None))
        
        if model != 'All' and m != model: continue
        if scheme != 'All' and s != scheme: continue
        if seed != 'All' and sd != seed: continue
        
        cols_to_plot.append(col)
    
    # --- PLOT ---
    plt.figure(figsize=(18, 8))
    
    # 1. Plot Actual Data (Black Line)
    plt.plot(sub_df.index, sub_df['actual'], 
             label='Actual Data', color='black', linewidth=2.5, alpha=0.9, zorder=100)
    
    # 2. Plot Models & Bands
    # Limit colors if too many lines are selected
    colors = sns.color_palette("bright", len(cols_to_plot))
    
    if not cols_to_plot:
        plt.text(0.5, 0.5, "No models match these filters", ha='center', transform=plt.gca().transAxes, fontsize=14)
    else:
        for i, col in enumerate(cols_to_plot):
            # Plot the Base Prediction Line
            plt.plot(sub_df.index, sub_df[col], label=col, color=colors[i], linewidth=2, alpha=1.0)
            
            # Plot the Uncertainty Band
            if show_band:
                col_L = f"{col}_L"
                col_U = f"{col}_U"
                
                # Verify bounds exist in the dataframe before plotting
                if col_L in sub_df.columns and col_U in sub_df.columns:
                    plt.fill_between(sub_df.index, 
                                     sub_df[col_L], 
                                     sub_df[col_U], 
                                     color=colors[i], 
                                     alpha=0.2, # Transparency for the band
                                     label='_nolegend_') # Exclude band from legend

    plt.title(f"Uncertainty Quantification Analysis: {len(cols_to_plot)} Models Selected", fontsize=16)
    plt.ylabel('Prediction Value')
    plt.xlabel('Date')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Smart Legend: Hide if too many items
    if len(cols_to_plot) <= 8:
        plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    else:
        plt.text(1.02, 0.95, f"Legend hidden\n({len(cols_to_plot)} models selected)\nFilter to < 8 to see details.", 
                 transform=plt.gca().transAxes, fontsize=11, verticalalignment='top')

    plt.tight_layout()
    plt.show()

# ==========================================
# 5. LAYOUT DISPLAY
# ==========================================
ui = widgets.VBox([
    widgets.HBox([w_model, w_scheme, w_seed, w_show_band]),
    w_range
])

out = widgets.interactive_output(plot_uq_series, {
    'model': w_model, 
    'scheme': w_scheme, 
    'seed': w_seed,
    'idx_range': w_range,
    'show_band': w_show_band
})

display(ui, out)

Output()